# Customer Churn Hive ETL
Notebook này dùng để kết nối Hive, chạy DDL/DML cho tầng raw và curated, và kiểm tra output churn analytics.

## 1. Cài thư viện trong notebook
Chạy cell cài package trước khi import. Bạn có thể chỉnh package theo cách kết nối Hive bạn chọn.

In [1]:
# Uncomment when ready to install libraries in the notebook kernel
%pip install pyhive thrift thrift-sasl pandas


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Cấu hình kết nối Hive

In [2]:
HIVE_HOST = 'dtwarehouse-hive'
HIVE_PORT = 10000
HIVE_USERNAME = 'root'
HIVE_DATABASE = 'default'
HIVE_AUTH = 'NONE'

## 3. Tạo kết nối Hive

In [3]:
import pandas as pd
from pyhive import hive

auth_candidates = []
if HIVE_AUTH is not None:
    auth_candidates.append(HIVE_AUTH)
if 'NONE' not in auth_candidates:
    auth_candidates.append('NONE')

last_error = None
for auth_mode in auth_candidates:
    try:
        kwargs = dict(
            host=HIVE_HOST,
            port=HIVE_PORT,
            username=HIVE_USERNAME,
            database=HIVE_DATABASE,
        )
        if auth_mode is not None:
            kwargs['auth'] = auth_mode
        conn = hive.Connection(**kwargs)
        print(f'Connected to Hive with user={HIVE_USERNAME}, auth={auth_mode}')

        # Stabilize local MR execution for parquet scans in this environment.
        session_settings = [
            'SET hive.vectorized.execution.enabled=false',
            'SET hive.vectorized.execution.reduce.enabled=false',
            'SET mapreduce.map.memory.mb=8192',
            'SET mapreduce.reduce.memory.mb=8192',
            "SET mapreduce.map.java.opts=-Xmx768m",
            "SET mapreduce.reduce.java.opts=-Xmx768m",
            "SET hive.auto.convert.join = false"
        ]
        cursor = conn.cursor()
        try:
            for stmt in session_settings:
                cursor.execute(stmt)
                print(f'Applied: {stmt}')
        finally:
            cursor.close()

        break
    except Exception as err:
        last_error = err
else:
    raise last_error

Connected to Hive with user=root, auth=NONE
Applied: SET hive.vectorized.execution.enabled=false
Applied: SET hive.vectorized.execution.reduce.enabled=false
Applied: SET mapreduce.map.memory.mb=8192
Applied: SET mapreduce.reduce.memory.mb=8192
Applied: SET mapreduce.map.java.opts=-Xmx768m
Applied: SET mapreduce.reduce.java.opts=-Xmx768m
Applied: SET hive.auto.convert.join = false


## 4. Kiểm tra kết nối

In [4]:
pd.read_sql('SHOW DATABASES', conn)

/tmp/ipykernel_5313/1387072856.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql('SHOW DATABASES', conn)


,database_name
0,curated_churn
1,default
2,raw_churn


## 5. Raw DDL cho churn data
Cell này chứa DDL để tạo external tables cho dữ liệu ingest từ HDFS.

In [5]:
raw_ddl = '''
CREATE DATABASE IF NOT EXISTS raw_churn;

CREATE EXTERNAL TABLE IF NOT EXISTS raw_churn.customers_raw (
  customer_id STRING,
  signup_date DATE,
  birth_year INT,
  gender STRING,
  city STRING,
  acquisition_channel STRING,
  segment STRING,
  is_active BOOLEAN
)
PARTITIONED BY (dt STRING)
STORED AS PARQUET
LOCATION 'hdfs://namenode:8020/data/raw/churn/customers';

CREATE EXTERNAL TABLE IF NOT EXISTS raw_churn.orders_raw (
  order_id STRING,
  customer_id STRING,
  order_ts TIMESTAMP,
  order_status STRING,
  currency STRING,
  subtotal_amount DECIMAL(18,2),
  discount_amount DECIMAL(18,2),
  shipping_fee DECIMAL(18,2),
  tax_amount DECIMAL(18,2),
  total_amount DECIMAL(18,2),
  payment_method STRING,
  promo_code STRING
)
PARTITIONED BY (dt STRING)
STORED AS PARQUET
LOCATION 'hdfs://namenode:8020/data/raw/churn/orders';
'''
print(raw_ddl)


CREATE DATABASE IF NOT EXISTS raw_churn;

CREATE EXTERNAL TABLE IF NOT EXISTS raw_churn.customers_raw (
  customer_id STRING,
  signup_date DATE,
  birth_year INT,
  gender STRING,
  city STRING,
  acquisition_channel STRING,
  segment STRING,
  is_active BOOLEAN
)
PARTITIONED BY (dt STRING)
STORED AS PARQUET
LOCATION 'hdfs://namenode:8020/data/raw/churn/customers';

CREATE EXTERNAL TABLE IF NOT EXISTS raw_churn.orders_raw (
  order_id STRING,
  customer_id STRING,
  order_ts TIMESTAMP,
  order_status STRING,
  currency STRING,
  subtotal_amount DECIMAL(18,2),
  discount_amount DECIMAL(18,2),
  shipping_fee DECIMAL(18,2),
  tax_amount DECIMAL(18,2),
  total_amount DECIMAL(18,2),
  payment_method STRING,
  promo_code STRING
)
PARTITIONED BY (dt STRING)
STORED AS PARQUET
LOCATION 'hdfs://namenode:8020/data/raw/churn/orders';



## 6. Curated SQL để derive churn labels và metrics

## 7. Helper để chạy nhiều câu lệnh SQL

In [6]:
def run_sql_script(connection, sql_script: str, script_name: str = "sql_script") -> None:
    statements = [s.strip() for s in sql_script.split(';') if s.strip()]
    cursor = connection.cursor()
    try:
        for statement in statements:
            preview = " ".join(statement.split())[:160]
            print(f"[{script_name}] Executing query: {preview}")
            try:
                cursor.execute(statement)
            except Exception as err:
                print(f"\n[{script_name}] FAILED")
                print("Statement text:")
                print(statement)
                raise RuntimeError(f"{script_name} failed") from err
    finally:
        cursor.close()

## 8. Chạy raw DDL / curated SQL

In [7]:
run_sql_script(conn, raw_ddl, script_name='raw_ddl')

[raw_ddl] Executing query: CREATE DATABASE IF NOT EXISTS raw_churn
[raw_ddl] Executing query: CREATE EXTERNAL TABLE IF NOT EXISTS raw_churn.customers_raw ( customer_id STRING, signup_date DATE, birth_year INT, gender STRING, city STRING, acquisition_chan
[raw_ddl] Executing query: CREATE EXTERNAL TABLE IF NOT EXISTS raw_churn.orders_raw ( order_id STRING, customer_id STRING, order_ts TIMESTAMP, order_status STRING, currency STRING, subtot


In [12]:
# Register HDFS partitions for external raw tables, then verify data is visible
run_sql_script(conn, """
MSCK REPAIR TABLE raw_churn.customers_raw;
MSCK REPAIR TABLE raw_churn.orders_raw;
""", script_name='raw_partition_repair')

print(pd.read_sql("SELECT COUNT(1) AS customers_cnt FROM raw_churn.customers_raw", conn))
# print(pd.read_sql("SELECT COUNT(1) AS orders_cnt FROM raw_churn.orders_raw", conn))

[raw_partition_repair] Executing query: MSCK REPAIR TABLE raw_churn.customers_raw
[raw_partition_repair] Executing query: MSCK REPAIR TABLE raw_churn.orders_raw


/tmp/ipykernel_5313/3732660976.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql("SELECT COUNT(1) AS customers_cnt FROM raw_churn.customers_raw", conn))


OperationalError: TExecuteStatementResp(status=TStatus(statusCode=3, infoMessages=['*org.apache.hive.service.cli.HiveSQLException:Error while processing statement: FAILED: Execution Error, return code 2 from org.apache.hadoop.hive.ql.exec.mr.MapRedTask:17:16', 'org.apache.hive.service.cli.operation.Operation:toSQLException:Operation.java:335', 'org.apache.hive.service.cli.operation.SQLOperation:runQuery:SQLOperation.java:226', 'org.apache.hive.service.cli.operation.SQLOperation:runInternal:SQLOperation.java:263', 'org.apache.hive.service.cli.operation.Operation:run:Operation.java:247', 'org.apache.hive.service.cli.session.HiveSessionImpl:executeStatementInternal:HiveSessionImpl.java:541', 'org.apache.hive.service.cli.session.HiveSessionImpl:executeStatement:HiveSessionImpl.java:516', 'org.apache.hive.service.cli.CLIService:executeStatement:CLIService.java:282', 'org.apache.hive.service.cli.thrift.ThriftCLIService:ExecuteStatement:ThriftCLIService.java:563', 'org.apache.hive.service.rpc

In [9]:
risk_tier_sql = '''


WITH latest_dt AS (
  SELECT MAX(dt) AS dt
  FROM curated_churn.customer_features_daily
)
SELECT
  f.customer_id,
  f.dt,
  CASE
    WHEN COALESCE(l.label_churn_30d, 0) = 1 THEN 0.90
    WHEN COALESCE(f.days_since_last_order, 0) >= 60 THEN 0.85
    WHEN COALESCE(f.days_since_last_order, 0) >= 45 THEN 0.75
    WHEN COALESCE(f.days_since_last_order, 0) >= 30 THEN 0.55
    WHEN COALESCE(f.orders_30d, 0) = 0 THEN 0.50
    ELSE 0.20
  END AS churn_risk_score,
  CASE
    WHEN (
      CASE
        WHEN COALESCE(l.label_churn_30d, 0) = 1 THEN 0.90
        WHEN COALESCE(f.days_since_last_order, 0) >= 60 THEN 0.85
        WHEN COALESCE(f.days_since_last_order, 0) >= 45 THEN 0.75
        WHEN COALESCE(f.days_since_last_order, 0) >= 30 THEN 0.55
        WHEN COALESCE(f.orders_30d, 0) = 0 THEN 0.50
        ELSE 0.20
      END
    ) >= 0.8 THEN 'High risk'
    WHEN (
      CASE
        WHEN COALESCE(l.label_churn_30d, 0) = 1 THEN 0.90
        WHEN COALESCE(f.days_since_last_order, 0) >= 60 THEN 0.85
        WHEN COALESCE(f.days_since_last_order, 0) >= 45 THEN 0.75
        WHEN COALESCE(f.days_since_last_order, 0) >= 30 THEN 0.55
        WHEN COALESCE(f.orders_30d, 0) = 0 THEN 0.50
        ELSE 0.20
      END
    ) >= 0.5 THEN 'Medium risk'
    ELSE 'Low risk'
  END AS risk_tier
FROM curated_churn.customer_features_daily f
LEFT JOIN curated_churn.churn_labels_daily l
  ON f.customer_id = l.customer_id
 AND f.dt = l.dt
JOIN latest_dt d
  ON f.dt = d.dt
ORDER BY churn_risk_score DESC, f.customer_id
LIMIT 100
'''

pd.read_sql(risk_tier_sql, conn)

/tmp/ipykernel_5313/1430229349.py:52: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(risk_tier_sql, conn)


OperationalError: TExecuteStatementResp(status=TStatus(statusCode=3, infoMessages=['*org.apache.hive.service.cli.HiveSQLException:Error while processing statement: FAILED: Execution Error, return code 2 from org.apache.hadoop.hive.ql.exec.mr.MapRedTask:17:16', 'org.apache.hive.service.cli.operation.Operation:toSQLException:Operation.java:335', 'org.apache.hive.service.cli.operation.SQLOperation:runQuery:SQLOperation.java:226', 'org.apache.hive.service.cli.operation.SQLOperation:runInternal:SQLOperation.java:263', 'org.apache.hive.service.cli.operation.Operation:run:Operation.java:247', 'org.apache.hive.service.cli.session.HiveSessionImpl:executeStatementInternal:HiveSessionImpl.java:541', 'org.apache.hive.service.cli.session.HiveSessionImpl:executeStatement:HiveSessionImpl.java:516', 'org.apache.hive.service.cli.CLIService:executeStatement:CLIService.java:282', 'org.apache.hive.service.cli.thrift.ThriftCLIService:ExecuteStatement:ThriftCLIService.java:563', 'org.apache.hive.service.rpc

## 9. Query kiểm tra output

In [10]:
pd.read_sql('SHOW TABLES IN curated_churn', conn)

/tmp/ipykernel_5313/2916706763.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql('SHOW TABLES IN curated_churn', conn)


,tab_name
0,churn_labels_daily
1,churn_metrics_daily
2,customer_features_daily


In [11]:
# Example verification query
pd.read_sql('SELECT * FROM curated_churn.churn_metrics_daily LIMIT 20', conn)

/tmp/ipykernel_5313/1066943404.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql('SELECT * FROM curated_churn.churn_metrics_daily LIMIT 20', conn)


,churn_metrics_daily.segment,churn_metrics_daily.city,churn_metrics_daily.customers_cnt,churn_metrics_daily.churn_30d_cnt,churn_metrics_daily.churn_60d_cnt,churn_metrics_daily.churn_30d_rate,churn_metrics_daily.churn_60d_rate,churn_metrics_daily.avg_revenue_90d,churn_metrics_daily.avg_orders_90d,churn_metrics_daily.dt
